# RiskWeave — Historical Daily Weather Collection (ERA5)

**Period:** 23 December 2006 – 19 December 2025  
**Source:** Open-Meteo Historical Weather API (ERA5)  
**Geography:** District representative coordinates from `data/reference/district_coordinates.csv`

This notebook collects daily environmental data separately for each district. Run the district cells **one at a time**. Each successful district is saved immediately as its own CSV and JSON file, so completed districts do not need to be downloaded again.

**Important:** HTTP 429 means the API is rate-limiting requests. This notebook does not wait for hours automatically. If a district receives 429, leave that district cell, wait for the indicated period, then rerun only that district cell. Do not repeatedly rerun it immediately.


## 1. Import libraries

In [ ]:
from pathlib import Path
import json
import time
import requests
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)


## 2. Set project paths and collection settings

In [ ]:
# This notebook is expected to be inside RiskWeave/notebooks/
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

COORDINATES_FILE = PROJECT_DIR / "data" / "reference" / "district_coordinates.csv"

OUTPUT_DIR = PROJECT_DIR / "data" / "raw" / "weather_era5_2006_2025"
DISTRICT_CSV_DIR = OUTPUT_DIR / "district_csv"
JSON_DIR = OUTPUT_DIR / "json"

DISTRICT_CSV_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2006-12-23"
END_DATE = "2025-12-19"
EXPECTED_DAYS = len(pd.date_range(START_DATE, END_DATE, freq="D"))

API_URL = "https://archive-api.open-meteo.com/v1/archive"
MODEL = "era5"
TIMEZONE = "Asia/Colombo"

DAILY_VARIABLES = [
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "wind_direction_10m_dominant",
    "cloud_cover_mean",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

print("Project folder:", PROJECT_DIR)
print("Coordinates file exists:", COORDINATES_FILE.exists())
print("Date range:", START_DATE, "to", END_DATE)
print("Expected daily rows per district:", EXPECTED_DAYS)
print("Output folder:", OUTPUT_DIR)


## 3. Load and inspect the 25 district coordinates

In [ ]:
if not COORDINATES_FILE.exists():
    raise FileNotFoundError(
        f"Coordinate file not found: {COORDINATES_FILE}. "
        "Check that data/reference/district_coordinates.csv exists."
    )

coordinates = pd.read_csv(COORDINATES_FILE)
print("Original columns:", coordinates.columns.tolist())

# Identify likely coordinate column names
column_lookup = {str(col).strip().lower(): col for col in coordinates.columns}

def find_column(possible_names, description):
    for name in possible_names:
        if name.lower() in column_lookup:
            return column_lookup[name.lower()]
    raise ValueError(
        f"Could not identify {description}. Available columns: {coordinates.columns.tolist()}"
    )

district_col = find_column(
    ["district", "area", "district_name", "name"], "district name column"
)
latitude_col = find_column(
    ["latitude", "lat", "representative_latitude"], "latitude column"
)
longitude_col = find_column(
    ["longitude", "lon", "lng", "representative_longitude"], "longitude column"
)

coordinates = coordinates[[district_col, latitude_col, longitude_col]].copy()
coordinates.columns = ["district", "latitude", "longitude"]

coordinates["district"] = coordinates["district"].astype(str).str.strip()
coordinates["latitude"] = pd.to_numeric(coordinates["latitude"], errors="coerce")
coordinates["longitude"] = pd.to_numeric(coordinates["longitude"], errors="coerce")
coordinates = coordinates.dropna(subset=["district", "latitude", "longitude"])
coordinates = coordinates.drop_duplicates(subset=["district"]).reset_index(drop=True)

print("Number of districts:", len(coordinates))
display(coordinates)


## 4. Define the district extraction function

In [ ]:
def collect_district(district_name):
    \"\"\"Download and save one district's daily ERA5 data.

    If the CSV is already complete, it is skipped.
    If Open-Meteo returns HTTP 429, the function reports Retry-After and exits
    without waiting for a long period. Rerun this district cell later.
    \"\"\"

    match = coordinates.loc[
        coordinates["district"].str.casefold() == district_name.casefold()
    ]

    if match.empty:
        available = coordinates["district"].tolist()
        raise ValueError(
            f"{district_name!r} not found in coordinate file. Available districts: {available}"
        )

    row = match.iloc[0]
    district = str(row["district"])
    safe_name = district.replace(" ", "_")
    csv_path = DISTRICT_CSV_DIR / f"{safe_name}.csv"
    json_path = JSON_DIR / f"{safe_name}.json"

    # Skip an already complete CSV
    if csv_path.exists():
        try:
            old = pd.read_csv(csv_path)
            old_dates = pd.to_datetime(old["date"], errors="coerce")
            is_complete = (
                len(old) == EXPECTED_DAYS
                and old_dates.notna().all()
                and old_dates.nunique() == EXPECTED_DAYS
                and old_dates.min() == pd.Timestamp(START_DATE)
                and old_dates.max() == pd.Timestamp(END_DATE)
                and old["district"].astype(str).eq(district).all()
                and all(v in old.columns for v in DAILY_VARIABLES)
            )
            if is_complete:
                print(f"SKIP {district}: complete CSV already exists ({len(old):,} rows)")
                return old
            else:
                print(f"Existing {district} CSV is incomplete or has a different schema; requesting again.")
        except Exception as exc:
            print(f"Could not validate existing CSV ({exc}); requesting again.")

    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(DAILY_VARIABLES),
        "timezone": TIMEZONE,
        "models": MODEL,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm",
    }

    print(f"Requesting {district} ...")
    try:
        response = requests.get(
            API_URL,
            params=params,
            timeout=180,
            headers={"User-Agent": "RiskWeave-Research/1.0"}
        )
    except requests.RequestException as exc:
        print("Request failed:", exc)
        return None

    if response.status_code == 429:
        retry_after = response.headers.get("Retry-After")
        print("HTTP 429: Open-Meteo rate limit reached.")
        print("Retry-After (seconds):", retry_after if retry_after else "not supplied")
        print("Do not rerun immediately. Wait, then rerun only this district's cell.")
        return None

    if not response.ok:
        print("HTTP error:", response.status_code)
        print(response.text[:1000])
        return None

    data = response.json()

    if "daily" not in data or "time" not in data["daily"]:
        print("The API response did not contain the expected daily data.")
        print(data)
        return None

    weather = pd.DataFrame(data["daily"]).rename(columns={"time": "date"})
    weather.insert(0, "district", district)

    # Keep requested representative point and returned ERA5 grid point separately.
    weather["requested_latitude"] = float(row["latitude"])
    weather["requested_longitude"] = float(row["longitude"])
    weather["era5_grid_latitude"] = data.get("latitude")
    weather["era5_grid_longitude"] = data.get("longitude")
    weather["era5_grid_elevation_m"] = data.get("elevation")
    weather["model"] = MODEL
    weather["timezone"] = TIMEZONE

    weather["date"] = pd.to_datetime(weather["date"]).dt.strftime("%Y-%m-%d")

    # Save CSV first so the district data is preserved immediately.
    weather.to_csv(csv_path, index=False)

    # Preserve the raw API response separately.
    with open(json_path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

    print(f"Saved {district}: {len(weather):,} rows")
    print("CSV:", csv_path)
    print("JSON:", json_path)

    if len(weather) != EXPECTED_DAYS:
        print(f"WARNING: expected {EXPECTED_DAYS:,} rows but received {len(weather):,}.")

    return weather


## 5. Extract each district separately

Run each code cell individually. If you receive HTTP 429, stop and wait before retrying that district. Do not run all 25 cells rapidly.

### Ampara

In [ ]:
weather_ampara = collect_district("Ampara")

### Anuradhapura

In [ ]:
weather_anuradhapura = collect_district("Anuradhapura")

### Badulla

In [ ]:
weather_badulla = collect_district("Badulla")

### Batticaloa

In [ ]:
weather_batticaloa = collect_district("Batticaloa")

### Colombo

In [ ]:
weather_colombo = collect_district("Colombo")

### Galle

In [ ]:
weather_galle = collect_district("Galle")

### Gampaha

In [ ]:
weather_gampaha = collect_district("Gampaha")

### Hambanthota

In [ ]:
weather_hambanthota = collect_district("Hambanthota")

### Jaffna

In [ ]:
weather_jaffna = collect_district("Jaffna")

### Kalmune

In [ ]:
weather_kalmune = collect_district("Kalmune")

### Kalutara

In [ ]:
weather_kalutara = collect_district("Kalutara")

### Kandy

In [ ]:
weather_kandy = collect_district("Kandy")

### Kegalle

In [ ]:
weather_kegalle = collect_district("Kegalle")

### Kilinochchi

In [ ]:
weather_kilinochchi = collect_district("Kilinochchi")

### Kurunegala

In [ ]:
weather_kurunegala = collect_district("Kurunegala")

### Mannar

In [ ]:
weather_mannar = collect_district("Mannar")

### Matale

In [ ]:
weather_matale = collect_district("Matale")

### Matara

In [ ]:
weather_matara = collect_district("Matara")

### Monaragala

In [ ]:
weather_monaragala = collect_district("Monaragala")

### Mullaitivu

In [ ]:
weather_mullaitivu = collect_district("Mullaitivu")

### NuwaraEliya

In [ ]:
weather_nuwaraeliya = collect_district("NuwaraEliya")

### Polonnaruwa

In [ ]:
weather_polonnaruwa = collect_district("Polonnaruwa")

### Puttalam

In [ ]:
weather_puttalam = collect_district("Puttalam")

### Ratnapura

In [ ]:
weather_ratnapura = collect_district("Ratnapura")

### Trincomalee

In [ ]:
weather_trincomalee = collect_district("Trincomalee")

### Vavuniya

In [ ]:
weather_vavuniya = collect_district("Vavuniya")

## 6. Validate the district CSV files

In [ ]:
validation_rows = []

for district in coordinates["district"]:
    safe_name = str(district).replace(" ", "_")
    csv_path = DISTRICT_CSV_DIR / f"{safe_name}.csv"

    if not csv_path.exists():
        validation_rows.append({
            "district": district,
            "file_exists": False,
            "rows": 0,
            "unique_dates": 0,
            "missing_values": None,
            "duplicate_dates": None,
            "date_min": None,
            "date_max": None,
            "status": "FILE MISSING"
        })
        continue

    df = pd.read_csv(csv_path)
    dates = pd.to_datetime(df["date"], errors="coerce")
    missing_values = int(df[DAILY_VARIABLES].isna().sum().sum())
    duplicate_dates = int(dates.duplicated().sum())

    complete = (
        len(df) == EXPECTED_DAYS
        and dates.notna().all()
        and dates.nunique() == EXPECTED_DAYS
        and dates.min() == pd.Timestamp(START_DATE)
        and dates.max() == pd.Timestamp(END_DATE)
        and missing_values == 0
        and duplicate_dates == 0
    )

    validation_rows.append({
        "district": district,
        "file_exists": True,
        "rows": len(df),
        "unique_dates": dates.nunique(),
        "missing_values": missing_values,
        "duplicate_dates": duplicate_dates,
        "date_min": dates.min().date() if dates.notna().any() else None,
        "date_max": dates.max().date() if dates.notna().any() else None,
        "status": "COMPLETE" if complete else "CHECK"
    })

validation = pd.DataFrame(validation_rows)
display(validation)
print("\nStatus counts:")
display(validation["status"].value_counts())


## 7. Combine complete district files into one master CSV

In [ ]:
complete_frames = []

for district in coordinates["district"]:
    safe_name = str(district).replace(" ", "_")
    csv_path = DISTRICT_CSV_DIR / f"{safe_name}.csv"

    if not csv_path.exists():
        continue

    df = pd.read_csv(csv_path)
    dates = pd.to_datetime(df["date"], errors="coerce")

    valid = (
        len(df) == EXPECTED_DAYS
        and dates.notna().all()
        and dates.nunique() == EXPECTED_DAYS
        and dates.min() == pd.Timestamp(START_DATE)
        and dates.max() == pd.Timestamp(END_DATE)
        and all(v in df.columns for v in DAILY_VARIABLES)
    )

    if valid:
        complete_frames.append(df)
    else:
        print(f"Not adding {district}: file is incomplete or invalid.")

if complete_frames:
    weather_master = pd.concat(complete_frames, ignore_index=True)
    master_path = OUTPUT_DIR / (
        f"weather_daily_era5_{START_DATE}_to_{END_DATE}.csv"
    )
    weather_master.to_csv(master_path, index=False)

    print("Master dataset saved:", master_path)
    print("Rows:", len(weather_master))
    print("Districts:", weather_master["district"].nunique())
    print("Date range:", weather_master["date"].min(), "to", weather_master["date"].max())
else:
    print("No complete district files are available to combine yet.")


## 8. Final checks for the master dataset

In [ ]:
if "weather_master" in globals():
    print("Shape:", weather_master.shape)
    print("Duplicate district-date records:",
          weather_master.duplicated(subset=["district", "date"]).sum())
    print("\nMissing values in daily weather variables:")
    display(weather_master[DAILY_VARIABLES].isna().sum().to_frame("missing_count"))
    print("\nRows per district:")
    display(weather_master.groupby("district").size().to_frame("daily_rows"))
else:
    print("Run section 7 after collecting districts to create the master dataset.")
